# Othello AI Agent — Minimax, Alpha–Beta & Expectimax

This notebook builds a complete **Othello (Reversi)** engine and several game-playing agents for this
**deterministic, perfect-information, two-player, zero-sum** game.

**What's inside**
1. A game engine: board, legal moves, disc flipping, passes and the end of the game.
2. Baseline agents: **Random** and **Greedy** (flips the most discs right away).
3. A hand-made **evaluation function** that combines corners, mobility and disc difference.
4. **Minimax** search with a depth limit.
5. **Alpha–Beta pruning**, with corner-first move ordering.
6. **Risk-aware Expectimax**, which models the opponent as one who sometimes blunders.
7. Experiments that compare nodes expanded, run time and results against the Greedy agent.
8. An interactive **Human vs AI** board built with `ipywidgets`.

**Conventions**
- Players: **Black = +1** moves first and **White = −1**.
- A move is a board coordinate `(row, col)`. A **pass** is `None`, and it's allowed only when the player has no legal move.
- The game ends when **neither player can move**, for example when the board is full.

## Rules summary

- Place a disc of your color on an **empty** square so that it **flanks** one or more contiguous enemy discs
  in at least one direction (horizontal, vertical or diagonal).
- Every flanked enemy disc in those directions is **flipped** to your color.
- A player with **no legal move** must **pass**. When **both** players must pass, the game is over.
- The player with **more discs** at the end wins. Ties are possible.

## 1. Imports

In [1]:
from __future__ import annotations

import math
import random
import time
from dataclasses import dataclass
from typing import Callable, List, Optional, Tuple

## 2. Game engine

The board is an 8×8 grid of integers: `1` is a black disc, `-1` a white disc and `0` an empty square.

In [2]:
N = 8                                   # board size (8x8)
BLACK, WHITE, EMPTY = 1, -1, 0
PLAYERS = {BLACK: "BLACK", WHITE: "WHITE"}

# The 8 directions a line of discs can be flanked in: (row step, col step)
DIRS = [(-1, -1), (-1, 0), (-1, 1),
        ( 0, -1),          ( 0, 1),
        ( 1, -1), ( 1, 0), ( 1, 1)]


def opponent(p: int) -> int:
    # Players are +1 / -1, so the opponent is just the negation
    return -p


def on_board(r: int, c: int) -> bool:
    return 0 <= r < N and 0 <= c < N


def initial_board() -> List[List[int]]:
    # Standard starting position: two discs of each color in the center
    b = [[EMPTY for _ in range(N)] for _ in range(N)]
    b[3][3] = WHITE; b[3][4] = BLACK
    b[4][3] = BLACK; b[4][4] = WHITE
    return b


def copy_board(b):
    return [row[:] for row in b]

In [3]:
def _flips_in_dir(board: List[List[int]], r: int, c: int, dr: int, dc: int, player: int) -> List[Tuple[int, int]]:
    # Walk from (r, c) in direction (dr, dc) and return the opponent discs that would be
    # flipped, or [] if the line is not closed by one of the player's own discs.
    flips = []
    r += dr; c += dc
    if not on_board(r, c) or board[r][c] != opponent(player):
        return []
    while on_board(r, c) and board[r][c] == opponent(player):
        flips.append((r, c))
        r += dr; c += dc
    if not on_board(r, c) or board[r][c] != player:
        return []
    return flips


def legal_moves(board: List[List[int]], player: int) -> List[Optional[Tuple[int, int]]]:
    # A move is legal if it is on an empty square and flips at least one disc.
    # If there is no legal move, the only option is to pass: [None].
    moves = []
    for r in range(N):
        for c in range(N):
            if board[r][c] != EMPTY:
                continue
            if any(_flips_in_dir(board, r, c, dr, dc, player) for dr, dc in DIRS):
                moves.append((r, c))
    return moves if moves else [None]


def apply_move(board: List[List[int]], move: Optional[Tuple[int, int]], player: int) -> List[List[int]]:
    # Return a NEW board after the move; the original board is not changed.
    if move is None:                    # pass: the board stays the same
        return copy_board(board)
    r, c = move
    assert board[r][c] == EMPTY, "illegal: target not empty"
    flips_all = []
    for dr, dc in DIRS:
        flips_all += _flips_in_dir(board, r, c, dr, dc, player)
    assert flips_all, "illegal: no flips created"
    nb = copy_board(board)
    nb[r][c] = player
    for fr, fc in flips_all:
        nb[fr][fc] = player
    return nb


def count_discs(board: List[List[int]]) -> Tuple[int, int, int]:
    # Return (black discs, white discs, empty squares)
    b = sum(cell == BLACK for row in board for cell in row)
    w = sum(cell == WHITE for row in board for cell in row)
    e = N * N - b - w
    return b, w, e

In [4]:
@dataclass(frozen=True)
class OthelloState:
    # Immutable game state: the board (as tuples) plus whose turn it is.
    board: Tuple[Tuple[int, ...], ...]
    player: int

    @staticmethod
    def from_board(board: List[List[int]], player: int) -> "OthelloState":
        return OthelloState(tuple(tuple(row) for row in board), player)

    def to_board(self) -> List[List[int]]:
        return [list(row) for row in self.board]


def is_terminal(state: OthelloState) -> bool:
    # The game is over only when NEITHER player has a legal move.
    b = state.to_board()
    if legal_moves(b, state.player) != [None]:
        return False
    if legal_moves(b, opponent(state.player)) != [None]:
        return False
    return True


def utility(state: OthelloState, perspective: int) -> float:
    # Result of a finished game from `perspective`'s point of view: +1 win, -1 loss, 0 draw.
    b, w, _ = count_discs(state.to_board())
    if b > w:
        winner = BLACK
    elif w > b:
        winner = WHITE
    else:
        return 0.0
    return 1.0 if winner == perspective else -1.0

In [5]:
def print_board(board: List[List[int]]):
    # Text view of the board: ● = black, ○ = white, . = empty
    header = "    " + " ".join(str(c) for c in range(N))
    print(header)
    print("   " + "--" * N)
    for r in range(N):
        row = []
        for c in range(N):
            v = board[r][c]
            row.append("●" if v == BLACK else "○" if v == WHITE else ".")
        print(f"{r:>2} | " + " ".join(row))
    b, w, e = count_discs(board)
    print(f"    (●=BLACK {b}, ○=WHITE {w}, empty {e})")


def pretty_move(m):
    return "pass" if m is None else f"({m[0]},{m[1]})"

## 3. Baseline agents and the game loop

Every agent has an `action(state)` method that returns a legal move, or `None` to pass.

In [6]:
class Agent:
    name = "Agent"

    def action(self, state: OthelloState) -> Optional[Tuple[int, int]]:
        raise NotImplementedError


class RandomAgent(Agent):
    # Plays a uniformly random legal move.
    name = "Random"

    def action(self, state: OthelloState):
        moves = legal_moves(state.to_board(), state.player)
        return random.choice(moves)


class GreedyFlipAgent(Agent):
    # Plays the move that flips the most discs right now (no look-ahead).
    def __init__(self):
        self.name = "GreedyFlips"

    def action(self, state: OthelloState):
        b = state.to_board()
        moves = legal_moves(b, state.player)
        if moves == [None]:
            return None
        best, best_flips = None, -1
        for r, c in moves:
            flips = sum(len(_flips_in_dir(b, r, c, dr, dc, state.player)) for dr, dc in DIRS)
            if flips > best_flips:
                best_flips, best = flips, (r, c)
        return best


def play_game(black_agent: Agent, white_agent: Agent, seed: Optional[int] = None, verbose: bool = False):
    # Play one full game. Returns (result, history) where result is
    # 1 = Black wins, -1 = White wins, 0 = draw.
    random.seed(seed)
    board = initial_board()
    state = OthelloState.from_board(board, BLACK)
    history = []
    if verbose:
        print_board(board)

    while not is_terminal(state):
        agent = black_agent if state.player == BLACK else white_agent
        move = agent.action(state)
        moves = legal_moves(state.to_board(), state.player)
        assert move in moves, f"{agent.name} chose illegal move {move}; legal are {moves}"

        if move is None:                                   # pass: only the turn changes
            history.append((state.player, None))
            state = OthelloState.from_board(state.to_board(), opponent(state.player))
            if verbose:
                print(f"{PLAYERS[opponent(state.player)]} passes.")
            continue

        new_board = apply_move(state.to_board(), move, state.player)
        history.append((state.player, move))
        state = OthelloState.from_board(new_board, opponent(state.player))
        if verbose:
            print(f"{PLAYERS[opponent(state.player)]} played {pretty_move(move)}")
            print_board(new_board)

    b, w, e = count_discs(state.to_board())
    result = 1 if b > w else -1 if w > b else 0
    if verbose:
        print("Final:")
        print_board(state.to_board())
        print("Draw." if result == 0 else f"Winner: {PLAYERS[result]}")
    return result, history

## 4. Evaluation function

Used at the depth limit to score positions that aren't finished yet (higher is better for `perspective`):

| Feature | Weight | Why |
|---|---|---|
| **Corners** (mine − opponent's) | 12 | A corner disc can never be flipped back, so corners are worth the most. |
| **Mobility** (my legal moves − opponent's) | 3 | More options means more control over the game. It is also more reliable than the disc count during the middle game. |
| **Disc difference** | 1 | Matters most at the end of the game, so it gets a small weight. |

In [7]:
CORNER_CELLS = [(0, 0), (0, 7), (7, 0), (7, 7)]


def evaluate(board: List[List[int]], perspective: int) -> float:
    # Heuristic score of `board` for `perspective` (BLACK=+1 or WHITE=-1).

    # 1) Disc difference
    b, w, e = count_discs(board)
    discs = (b - w) if perspective == BLACK else (w - b)

    # 2) Mobility: number of legal moves for each side (a pass counts as 0 moves)
    def move_count(moves):
        return 0 if moves == [None] else len(moves)

    mobility = (move_count(legal_moves(board, perspective))
                - move_count(legal_moves(board, opponent(perspective))))

    # 3) Corner control: corners can never be flipped back
    my_corners = sum(board[r][c] == perspective for r, c in CORNER_CELLS)
    opp_corners = sum(board[r][c] == opponent(perspective) for r, c in CORNER_CELLS)
    corner_score = my_corners - opp_corners

    # Corners matter most, then mobility, then the raw disc count
    return 12 * corner_score + 3 * mobility + 1 * discs

## 5. Minimax

- The search goes down to a fixed **depth**, measured in plies (one ply is one player's move).
- The current player is the **maximizer** and the opponent is the **minimizer**.
- Finished games are scored with `utility`. Positions at the depth limit are scored with `evaluate`.
- A pass counts as a ply: the turn changes and the board stays the same.
- `self.nodes` counts every node the search visits.

In [8]:
class MinimaxAgent(Agent):
    def __init__(self, depth: int = 4, eval_fn: Callable[[List[List[int]], int], float] = evaluate):
        self.depth = depth
        self.eval_fn = eval_fn
        self.name = f"Minimax(d={depth})"
        self.nodes = 0                      # number of nodes visited during search

    def action(self, state: OthelloState) -> Optional[Tuple[int, int]]:
        perspective = state.player
        board = state.to_board()
        moves = legal_moves(board, state.player)
        if moves == [None]:
            return None                     # no legal move: pass

        # Root = MAX node: try every move and keep the one with the highest minimax value
        best_move, best_val = None, -math.inf
        for m in moves:
            child = OthelloState.from_board(apply_move(board, m, state.player), opponent(state.player))
            v = self._min_value(child, self.depth - 1, perspective)
            if v > best_val:
                best_val, best_move = v, m
        return best_move

    def _max_value(self, state: OthelloState, depth: int, perspective: int) -> float:
        # Our turn: pick the child with the highest value
        self.nodes += 1
        if is_terminal(state):
            return utility(state, perspective)
        if depth <= 0:                      # depth limit reached: use the heuristic
            return self.eval_fn(state.to_board(), perspective)

        b = state.to_board()
        moves = legal_moves(b, state.player)
        if moves == [None]:                 # we must pass: the turn goes to the opponent
            return self._min_value(OthelloState.from_board(b, opponent(state.player)), depth - 1, perspective)

        value = -math.inf
        for m in moves:
            child = OthelloState.from_board(apply_move(b, m, state.player), opponent(state.player))
            value = max(value, self._min_value(child, depth - 1, perspective))
        return value

    def _min_value(self, state: OthelloState, depth: int, perspective: int) -> float:
        # Opponent's turn: they pick the child with the lowest value for us
        self.nodes += 1
        if is_terminal(state):
            return utility(state, perspective)
        if depth <= 0:
            return self.eval_fn(state.to_board(), perspective)

        b = state.to_board()
        moves = legal_moves(b, state.player)
        if moves == [None]:                 # opponent must pass: the turn comes back to us
            return self._max_value(OthelloState.from_board(b, opponent(state.player)), depth - 1, perspective)

        value = math.inf
        for m in moves:
            child = OthelloState.from_board(apply_move(b, m, state.player), opponent(state.player))
            value = min(value, self._max_value(child, depth - 1, perspective))
        return value

## 6. Alpha–Beta pruning

The search is the same as Minimax and returns the **same values**, but it skips branches that can't change the decision:
- `alpha` is the best value the maximizer can already guarantee, and `beta` is the best value the minimizer can already guarantee.
- At a MAX node, the search stops when `value ≥ beta`. At a MIN node, it stops when `value ≤ alpha`.
- **Move ordering:** corner moves are tried first. Strong moves found early mean more pruning later.

In [9]:
def order_moves(moves):
    # Try corner moves first: they are usually strong and lead to earlier cutoffs
    return sorted(moves, key=lambda m: 1 if m in CORNER_CELLS else 0, reverse=True)


class AlphaBetaAgent(Agent):
    def __init__(self, depth: int = 5, eval_fn: Callable[[List[List[int]], int], float] = evaluate):
        self.depth = depth
        self.eval_fn = eval_fn
        self.name = f"AlphaBeta(d={depth})"
        self.nodes = 0

    def action(self, state: OthelloState) -> Optional[Tuple[int, int]]:
        perspective = state.player
        b = state.to_board()
        moves = legal_moves(b, state.player)
        if moves == [None]:
            return None

        alpha, beta = -math.inf, math.inf
        best_move, best_val = None, -math.inf
        for m in order_moves(moves):
            child = OthelloState.from_board(apply_move(b, m, state.player), opponent(state.player))
            v = self._min_value(child, self.depth - 1, perspective, alpha, beta)
            if v > best_val:
                best_val, best_move = v, m
            alpha = max(alpha, best_val)    # best value guaranteed at the root so far
        return best_move

    def _max_value(self, state: OthelloState, depth: int, perspective: int, alpha: float, beta: float) -> float:
        self.nodes += 1
        if is_terminal(state):
            return utility(state, perspective)
        if depth <= 0:
            return self.eval_fn(state.to_board(), perspective)

        b = state.to_board()
        moves = legal_moves(b, state.player)
        if moves == [None]:                 # pass: the turn goes to the opponent
            return self._min_value(OthelloState.from_board(b, opponent(state.player)),
                                   depth - 1, perspective, alpha, beta)

        value = -math.inf
        for m in order_moves(moves):
            child = OthelloState.from_board(apply_move(b, m, state.player), opponent(state.player))
            value = max(value, self._min_value(child, depth - 1, perspective, alpha, beta))
            alpha = max(alpha, value)
            if value >= beta:               # beta cutoff: MIN will never let the game reach here
                break
        return value

    def _min_value(self, state: OthelloState, depth: int, perspective: int, alpha: float, beta: float) -> float:
        self.nodes += 1
        if is_terminal(state):
            return utility(state, perspective)
        if depth <= 0:
            return self.eval_fn(state.to_board(), perspective)

        b = state.to_board()
        moves = legal_moves(b, state.player)
        if moves == [None]:                 # pass: the turn comes back to us
            return self._max_value(OthelloState.from_board(b, opponent(state.player)),
                                   depth - 1, perspective, alpha, beta)

        value = math.inf
        for m in order_moves(moves):
            child = OthelloState.from_board(apply_move(b, m, state.player), opponent(state.player))
            value = min(value, self._max_value(child, depth - 1, perspective, alpha, beta))
            beta = min(beta, value)
            if value <= alpha:              # alpha cutoff: MAX already has a better option
                break
        return value

## 7. Risk-aware Expectimax (imperfect opponent)

The opponent is modeled as a **chance node** instead of a perfect minimizer:
- With probability **1 − ε**, the opponent plays its **best** move, the one that minimizes our value.
- With probability **ε**, it **blunders** and plays one of its other legal moves, chosen uniformly at random.

$$V_{\text{opp}} = (1-\varepsilon)\,v_{\text{best}} + \frac{\varepsilon}{K}\sum_{\text{other moves}} v_m$$

$K$ is the number of other moves. Because of this, the agent sometimes takes a move that strict minimax
would call risky, when its **expected value** is higher because the opponent may make a mistake.

In [10]:
class ExpectimaxRiskyAgent(Agent):
    # Expectimax with an imperfect opponent.
    #   depth:        search depth in plies
    #   eval_fn:      evaluation(board, perspective) -> float
    #   blunder_rate: epsilon in [0, 1], the probability that the opponent does not play its best move
    def __init__(self,
                 depth: int = 4,
                 eval_fn: Callable[[List[List[int]], int], float] = evaluate,
                 blunder_rate: float = 0.20):
        self.depth = depth
        self.eval_fn = eval_fn
        self.eps = max(0.0, min(1.0, blunder_rate))
        self.name = f"ExpectimaxRisky(d={depth}, eps={self.eps:.2f})"
        self.nodes = 0

    def action(self, state: OthelloState) -> Optional[Tuple[int, int]]:
        # Pick the move with the highest EXPECTED value under the imperfect-opponent model
        board = state.to_board()
        moves = legal_moves(board, state.player)
        if moves == [None]:
            return None

        perspective = state.player
        best_move, best_val = None, -math.inf
        for m in moves:
            child = OthelloState.from_board(apply_move(board, m, state.player), opponent(state.player))
            v = self._opponent_expectation(child, self.depth - 1, perspective)
            if v > best_val:
                best_val, best_move = v, m
        return best_move

    def _max_value(self, state: OthelloState, depth: int, perspective: int) -> float:
        # Our turn: a normal MAX node
        self.nodes += 1
        if is_terminal(state):
            return utility(state, perspective)
        if depth <= 0:
            return self.eval_fn(state.to_board(), perspective)

        b = state.to_board()
        moves = legal_moves(b, state.player)
        if moves == [None]:                 # pass: the turn goes to the opponent
            return self._opponent_expectation(OthelloState.from_board(b, opponent(state.player)),
                                              depth - 1, perspective)

        value = -math.inf
        for m in moves:
            child = OthelloState.from_board(apply_move(b, m, state.player), opponent(state.player))
            value = max(value, self._opponent_expectation(child, depth - 1, perspective))
        return value

    def _opponent_expectation(self, state: OthelloState, depth: int, perspective: int) -> float:
        # Opponent's turn as a CHANCE node:
        #   best move (lowest value for us) with probability 1 - eps,
        #   each of the other K moves with probability eps / K.
        self.nodes += 1
        if is_terminal(state):
            return utility(state, perspective)
        if depth <= 0:
            return self.eval_fn(state.to_board(), perspective)

        b = state.to_board()
        moves = legal_moves(b, state.player)
        if moves == [None]:                 # opponent must pass: the turn comes back to us
            return self._max_value(OthelloState.from_board(b, opponent(state.player)), depth - 1, perspective)

        # Value of each opponent move (it will be our turn afterwards)
        vals = []
        for m in moves:
            child = OthelloState.from_board(apply_move(b, m, state.player), opponent(state.player))
            vals.append(self._max_value(child, depth - 1, perspective))

        # The opponent's best move is the one that minimizes our value
        vals.sort()
        v_best, others = vals[0], vals[1:]

        # With one legal move (or eps = 0) the opponent is deterministic
        if not others or self.eps <= 1e-12:
            return v_best

        # Expected value
        return (1.0 - self.eps) * v_best + (self.eps / len(others)) * sum(others)

## 8. Engine checks

These cells check that the game rules work correctly. They don't test the search agents.

In [11]:
b = initial_board()
print_board(b)

# Each side has exactly 4 legal opening moves
assert len([m for m in legal_moves(b, BLACK) if m is not None]) == 4
assert len([m for m in legal_moves(b, WHITE) if m is not None]) == 4

# Black plays (2,3) and flips the disc at (3,3)
assert (2, 3) in legal_moves(b, BLACK)
b2 = apply_move(b, (2, 3), BLACK)
print_board(b2)

print("✓ Basic engine checks passed.")

    0 1 2 3 4 5 6 7
   ----------------
 0 | . . . . . . . .
 1 | . . . . . . . .
 2 | . . . . . . . .
 3 | . . . ○ ● . . .
 4 | . . . ● ○ . . .
 5 | . . . . . . . .
 6 | . . . . . . . .
 7 | . . . . . . . .
    (●=BLACK 2, ○=WHITE 2, empty 60)
    0 1 2 3 4 5 6 7
   ----------------
 0 | . . . . . . . .
 1 | . . . . . . . .
 2 | . . . ● . . . .
 3 | . . . ● ● . . .
 4 | . . . ● ○ . . .
 5 | . . . . . . . .
 6 | . . . . . . . .
 7 | . . . . . . . .
    (●=BLACK 4, ○=WHITE 1, empty 59)
✓ Basic engine checks passed.


## 9. Experiments

Each search agent plays **Black** against the **Greedy** agent (White). For every agent we record the result,
the total number of nodes expanded and the game time.

Set `verbose=True` in `play_game` to print the board after every move.

In [12]:
# Baseline: Random vs Greedy
res, _ = play_game(RandomAgent(), GreedyFlipAgent(), seed=0)
print("Random (Black) vs GreedyFlips (White):", res)

Random (Black) vs GreedyFlips (White): -1


In [13]:
def run_match(agent, seed=2):
    # Play `agent` (Black) against Greedy (White) and return a summary row
    start = time.perf_counter()
    res, _ = play_game(agent, GreedyFlipAgent(), seed=seed)
    elapsed = time.perf_counter() - start
    winner = {1: "agent", -1: "Greedy", 0: "draw"}[res]
    return agent.name, winner, agent.nodes, elapsed


print(f"{'Agent (Black) vs Greedy (White)':<34} {'Winner':<8} {'Nodes':>8} {'Time (s)':>9}")
print("-" * 62)
for depth in (2, 3):
    for agent in (MinimaxAgent(depth=depth),
                  AlphaBetaAgent(depth=depth),
                  ExpectimaxRiskyAgent(depth=depth, eval_fn=evaluate, blunder_rate=0.3)):
        name, winner, nodes, elapsed = run_match(agent)
        print(f"{name:<34} {winner:<8} {nodes:>8} {elapsed:>9.2f}")

Agent (Black) vs Greedy (White)    Winner      Nodes  Time (s)
--------------------------------------------------------------
Minimax(d=2)                       agent        1729      0.48
AlphaBeta(d=2)                     agent         849      0.19
ExpectimaxRisky(d=2, eps=0.30)     agent        1729      0.40
Minimax(d=3)                       agent       19486      4.95
AlphaBeta(d=3)                     agent        5084      1.19
ExpectimaxRisky(d=3, eps=0.30)     agent       19667      5.13


**Observations**
- **Alpha–Beta** picks exactly the same moves as **Minimax** at the same depth, but it expands far fewer nodes and runs faster.
- **Expectimax** can't prune, because every opponent move contributes to the expected value. It expands the most nodes.
- The search cost grows **exponentially** with depth. In the middle of the game the branching factor is largest, and Othello positions often have more than 10 legal moves.

## 10. Human vs AI

Play against any of the agents in an interactive board. This needs **`ipywidgets`**:

```bash
pip install ipywidgets
```

On Windows with the classic Notebook, you may also need:
```bash
jupyter nbextension enable --py widgetsnbextension --sys-prefix
```

Run the cell below to check your setup, then run the next cell to start the game.

In [ ]:
import platform
import subprocess
import sys


def install_and_setup_widgets():
    # Install and enable ipywidgets for Jupyter if needed
    print("Checking ipywidgets installation...")
    try:
        import ipywidgets as widgets
        print("✓ ipywidgets is already installed")
    except ImportError:
        print("Installing ipywidgets...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "ipywidgets", "-q"])
        print("✓ ipywidgets installed")

    if platform.system() == "Windows":
        print("\nEnabling Jupyter widget extensions for Windows...")
        try:
            subprocess.check_call([sys.executable, "-m", "jupyter", "nbextension", "enable",
                                   "--py", "widgetsnbextension", "--sys-prefix"],
                                  stderr=subprocess.DEVNULL)
            print("✓ Extension enabled")
        except Exception:
            print("Note: run this manually if needed:")
            print("  jupyter nbextension enable --py widgetsnbextension --sys-prefix")

    print("\n✓ Setup complete! Refresh the page or restart Jupyter if the widgets don't appear.")


install_and_setup_widgets()

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML
    WIDGETS_AVAILABLE = True
except ImportError as e:
    print("ERROR: ipywidgets is not installed or not working. Run the setup cell above first.")
    print(f"Error details: {e}")
    WIDGETS_AVAILABLE = False

if not WIDGETS_AVAILABLE:
    raise ImportError("Cannot start GUI without ipywidgets")


class OthelloGUI:
    # Clickable Othello board: choose an agent and a color, then play against the AI.

    # Menu option -> function that builds the agent
    AGENTS = {
        'random':       lambda: RandomAgent(),
        'greedy':       lambda: GreedyFlipAgent(),
        'minimax_3':    lambda: MinimaxAgent(depth=3),
        'minimax_4':    lambda: MinimaxAgent(depth=4),
        'alphabeta_3':  lambda: AlphaBetaAgent(depth=3),
        'alphabeta_4':  lambda: AlphaBetaAgent(depth=4),
        'alphabeta_5':  lambda: AlphaBetaAgent(depth=5),
        'expectimax_3': lambda: ExpectimaxRiskyAgent(depth=3, blunder_rate=0.2),
        'expectimax_4': lambda: ExpectimaxRiskyAgent(depth=4, blunder_rate=0.2),
    }

    def __init__(self):
        self.state = None
        self.agent = None
        self.human_color = None
        self.game_active = False
        self.last_ai_move = None
        self.cell_size = 60                 # button size in pixels
        self.setup_menu()

    # ---------- Menu ----------
    def setup_menu(self):
        clear_output(wait=True)
        title = widgets.HTML("<h2>🎮 Othello: Human vs AI</h2>")

        agent_dropdown = widgets.Dropdown(
            options=[
                ('Random', 'random'),
                ('Greedy Flips', 'greedy'),
                ('Minimax (depth=3)', 'minimax_3'),
                ('Minimax (depth=4)', 'minimax_4'),
                ('Alpha-Beta (depth=3)', 'alphabeta_3'),
                ('Alpha-Beta (depth=4)', 'alphabeta_4'),
                ('Alpha-Beta (depth=5)', 'alphabeta_5'),
                ('Expectimax (depth=3, ε=0.2)', 'expectimax_3'),
                ('Expectimax (depth=4, ε=0.2)', 'expectimax_4'),
            ],
            value='alphabeta_4',
            description='AI Agent:',
            style={'description_width': '100px'}
        )
        color_dropdown = widgets.Dropdown(
            options=[('Black (●) - Go First', BLACK), ('White (○) - Go Second', WHITE)],
            value=BLACK,
            description='Your Color:',
            style={'description_width': '100px'}
        )
        start_button = widgets.Button(description='Start Game', button_style='success', icon='play')

        def on_start_clicked(_):
            self.agent = self.AGENTS[agent_dropdown.value]()
            self.human_color = color_dropdown.value
            self.start_game()

        start_button.on_click(on_start_clicked)
        display(widgets.VBox([title, agent_dropdown, color_dropdown, start_button]))

    # ---------- Game setup ----------
    def start_game(self):
        clear_output(wait=True)
        self.state = OthelloState.from_board(initial_board(), BLACK)
        self.game_active = True
        self.last_ai_move = None
        self.create_game_ui()
        if self.state.player != self.human_color:   # the AI moves first if the human plays White
            self.make_ai_move()

    def create_game_ui(self):
        self.info_label = widgets.HTML()
        self.buttons = []
        rows = []

        # Round, disc-like buttons
        display(HTML("""
        <style>
        .othello-cell button {
            border-radius: 50% !important;
            font-size: 24px !important;
            font-weight: bold !important;
            color: white !important;
            padding: 0 !important;
            transition: all 0.2s ease;
            box-shadow: inset 0 2px 4px rgba(0,0,0,0.2);
        }
        .othello-cell button:hover:not(:disabled) {
            transform: scale(1.15);
            box-shadow: 0 6px 12px rgba(0,0,0,0.4), inset 0 2px 4px rgba(0,0,0,0.2);
            cursor: pointer;
        }
        </style>
        """))

        # 8x8 grid of buttons, one per square
        for r in range(N):
            row = []
            for c in range(N):
                btn = widgets.Button(
                    description='',
                    layout=widgets.Layout(width=f'{self.cell_size}px', height=f'{self.cell_size}px', margin='2px')
                )
                btn.add_class('othello-cell')
                btn.on_click(lambda b, row=r, col=c: self.on_cell_click(row, col))
                row.append(btn)
            rows.append(widgets.HBox(row, layout=widgets.Layout(margin='0')))
            self.buttons.append(row)

        board_box = widgets.VBox(rows, layout=widgets.Layout(
            margin='10px 0px', padding='15px', background='#1b5e20', border_radius='10px'))

        self.pass_button = widgets.Button(description='Pass', button_style='warning', icon='forward',
                                          layout=widgets.Layout(width='150px'))
        self.pass_button.on_click(lambda b: self.on_pass())

        new_game_button = widgets.Button(description='New Game', button_style='info', icon='refresh',
                                         layout=widgets.Layout(width='150px'))
        new_game_button.on_click(lambda b: self.setup_menu())

        self.status_output = widgets.Output()
        display(widgets.VBox([self.info_label, board_box,
                              widgets.HBox([self.pass_button, new_game_button]),
                              self.status_output]))
        self.update_display()

    # ---------- Drawing ----------
    def update_display(self):
        board = self.state.to_board()
        legal = legal_moves(board, self.state.player) if self.game_active else []

        for r in range(N):
            for c in range(N):
                btn = self.buttons[r][c]
                cell = board[r][c]
                btn.description = ''
                btn.disabled = True
                is_last_move = self.last_ai_move and (r, c) == self.last_ai_move
                border = '3px solid #FFA500' if is_last_move else '2px solid #888'   # orange = AI's last move

                if cell == BLACK:
                    btn.style.button_color = '#000000'
                    btn.layout.border = border
                elif cell == WHITE:
                    btn.style.button_color = '#F5F5DC'
                    btn.layout.border = border
                elif self.game_active and self.state.player == self.human_color and (r, c) in legal:
                    # Legal move for the human: green "+" and clickable
                    btn.description = '+'
                    btn.style.button_color = '#28a745'
                    btn.layout.border = '2px solid #1e7e34'
                    btn.disabled = False
                else:
                    btn.style.button_color = '#2d6a4f'
                    btn.layout.border = '2px solid #1b4332'

        # Score and status panel
        b, w, e = count_discs(board)
        info_html = f"""
        <div style='font-size: 18px; margin-bottom: 10px; padding: 10px; background: #f0f0f0; border-radius: 5px;'>
            <b>⚫ Black:</b> {b} | <b>⚪ White:</b> {w} | <b>Empty:</b> {e}<br>
        """
        if self.game_active:
            current_player = "⚫ Black" if self.state.player == BLACK else "⚪ White"
            if self.state.player == self.human_color:
                info_html += f"<b style='color: green;'>🎯 Your turn ({current_player})</b><br>"
                if self.last_ai_move:
                    info_html += (f"<small style='color: #ff8800;'>🔶 Orange = AI's last move at "
                                  f"({self.last_ai_move[0]}, {self.last_ai_move[1]})</small>")
            else:
                info_html += f"<b style='color: blue;'>🤖 AI's turn ({current_player})</b>"
        else:
            winner = "⚫ Black" if b > w else "⚪ White" if w > b else "🤝 Draw"
            info_html += f"<b style='color: red;'>🏁 Game Over! Winner: {winner}</b>"
        info_html += "</div>"
        self.info_label.value = info_html

        # The Pass button is only enabled when the human has no legal move
        self.pass_button.disabled = not (self.game_active and self.state.player == self.human_color
                                         and legal == [None])

    # ---------- Human actions ----------
    def on_cell_click(self, row, col):
        if not self.game_active or self.state.player != self.human_color:
            return
        board = self.state.to_board()
        if (row, col) not in legal_moves(board, self.state.player):
            return

        self.state = OthelloState.from_board(apply_move(board, (row, col), self.state.player),
                                             opponent(self.state.player))
        self.last_ai_move = None
        with self.status_output:
            clear_output(wait=True)
            print(f"You played ({row}, {col})")
        self.after_human_turn()

    def on_pass(self):
        if not self.game_active or self.state.player != self.human_color:
            return
        board = self.state.to_board()
        if legal_moves(board, self.state.player) != [None]:
            return

        self.state = OthelloState.from_board(board, opponent(self.state.player))
        self.last_ai_move = None
        with self.status_output:
            clear_output(wait=True)
            print("You passed.")
        self.after_human_turn()

    def after_human_turn(self):
        self.update_display()
        if is_terminal(self.state):
            self.end_game()
        elif self.state.player != self.human_color:
            self.make_ai_move()

    # ---------- AI move ----------
    def make_ai_move(self):
        time.sleep(0.3)                     # short pause so the human can see the board change
        board = self.state.to_board()

        if legal_moves(board, self.state.player) == [None]:
            self.state = OthelloState.from_board(board, opponent(self.state.player))
            self.last_ai_move = None
            with self.status_output:
                clear_output(wait=True)
                print(f"AI ({self.agent.name}) passed.")
        else:
            move = self.agent.action(self.state)
            self.state = OthelloState.from_board(apply_move(board, move, self.state.player),
                                                 opponent(self.state.player))
            self.last_ai_move = move
            with self.status_output:
                clear_output(wait=True)
                print(f"AI ({self.agent.name}) played ({move[0]}, {move[1]})")

        self.update_display()
        if is_terminal(self.state):
            self.end_game()

    # ---------- End of game ----------
    def end_game(self):
        self.game_active = False
        self.update_display()
        b, w, e = count_discs(self.state.to_board())
        with self.status_output:
            clear_output(wait=True)
            if b == w:
                print(f"🤝 Draw! Black: {b}, White: {w}")
            elif (b > w) == (self.human_color == BLACK):
                print(f"🎉 You won! Black: {b}, White: {w}")
            else:
                print(f"😞 AI won! Black: {b}, White: {w}")


gui = OthelloGUI()